In [9]:
import os, re
import numpy as np
import pandas as pd

In [10]:
def clean_text(x):
    """清理 Excel 表头中的多余空格。"""
    if pd.isna(x):
        return ""
    x = str(x).replace("\u3000", " ").strip()
    x = re.sub(r"\s+", " ", x)
    return x


def load_rice_phenotype_excel(xlsx_path, sheet_name="Sheet1"):
    """
    读取该水稻表型 Excel。

    返回结果：
    - df 的列是两层 MultiIndex：
      第一层：region，例如 meta / Shanghai / Hangzhou / Hainan
      第二层：trait，例如 Population / ID / Heading date (d)
    """

    raw = pd.read_excel(
        xlsx_path,
        sheet_name=sheet_name,
        header=[0, 1],
        na_values=["NA", ""],
        keep_default_na=True,
        engine="openpyxl"
    )

    new_cols = []
    last_region = None

    for i, (region, trait) in enumerate(raw.columns):
        if i == 0:
            new_cols.append(("meta", "Population"))
        elif i == 1:
            new_cols.append(("meta", "ID"))
        else:
            region = clean_text(region)
            trait = clean_text(trait)
            
            # 去除性状名末尾的单位，例如 Heading date (d) -> Heading date
            trait = re.sub(r"\s*\([^)]*\)\s*$", "", trait).strip()

            if region.startswith("Unnamed") or region == "":
                region = last_region
            else:
                last_region = region

            new_cols.append((region, trait))

    raw.columns = pd.MultiIndex.from_tuples(
        new_cols,
        names=["region", "trait"]
    )

    # Population 列原始 Excel 中是合并单元格，pandas 读入后只有第一行有值，需要向下填充
    raw[("meta", "Population")] = raw[("meta", "Population")].ffill()

    # ID 去除可能存在的前后空格
    raw[("meta", "ID")] = raw[("meta", "ID")].astype(str).str.strip()

    # 删除全空行
    raw = raw.dropna(how="all")

    return raw

def available_populations(df):
    """查看所有亲本 / 群体名称。"""
    return df[("meta", "Population")].dropna().drop_duplicates().tolist()

def available_regions(df):
    """查看所有地域。"""
    return [
        x for x in df.columns.get_level_values("region").unique()
        if x != "meta"
    ]

def available_traits(df, region=None):
    """
    查看性状名。
    如果指定 region，则只返回该地域下的性状。
    """
    if region is None:
        out = {}
        for r in available_regions(df):
            out[r] = list(df.loc[:, r].columns)
        return out

    region = match_one(available_regions(df), region, "region")
    return list(df.loc[:, region].columns)

def match_one(candidates, key, name="key"):
    """
    支持大小写不敏感和部分匹配。
    例如 region='shanghai' 可以匹配 Shanghai；
    trait='heading date' 可以匹配 Heading date (d)。
    """
    key_norm = clean_text(key).lower()

    mapping = {
        clean_text(c).lower(): c
        for c in candidates
    }

    if key_norm in mapping:
        return mapping[key_norm]

    hits = [
        c for c in candidates
        if key_norm in clean_text(c).lower()
    ]

    if len(hits) == 1:
        return hits[0]

    raise KeyError(
        f"无法唯一匹配 {name}={key!r}。候选值为：{list(candidates)}；匹配结果为：{hits}"
    )

def get_common_numeric_phenotypes(df, regions=("Shanghai", "Hangzhou", "Hainan")):
    """
    获取 Shanghai、Hangzhou、Hainan 三个地域共同存在的数值型表型列。
    返回结果仍然保留 MultiIndex 列结构，并保留 meta/Population 和 meta/ID。
    """

    first_traits = list(df.loc[:, regions[0]].columns)

    common_traits = [
        trait for trait in first_traits
        if all(trait in df.loc[:, region].columns for region in regions)
    ]

    numeric_traits = []

    for trait in common_traits:
        is_numeric = True

        for region in regions:
            s = df[(region, trait)]
            non_missing = s.notna()
            s_num = pd.to_numeric(s, errors="coerce")

            if not non_missing.any() or not s_num[non_missing].notna().all():
                is_numeric = False
                break

        if is_numeric:
            numeric_traits.append(trait)

    cols = [("meta", "Population"), ("meta", "ID")]
    cols += [(region, trait) for region in regions for trait in numeric_traits]

    out = df.loc[:, cols].copy()

    for region in regions:
        for trait in numeric_traits:
            out[(region, trait)] = pd.to_numeric(out[(region, trait)], errors="coerce")

    return out

def select_populations(df, populations=None):
    """
    选择全部品种，或选择某几个亲本对应的所有品种。

    populations=None 或 populations="all"：返回全部品种
    populations="某个亲本名"：返回该亲本下所有品种
    populations=["亲本1", "亲本2"]：返回多个亲本下所有品种
    """

    if populations is None or populations == "all":
        return df.copy()

    if isinstance(populations, str):
        populations = [populations]

    mask = df[("meta", "Population")].isin(populations)

    return df.loc[mask].copy()

def to_region_long(df, regions=("Shanghai", "Hangzhou", "Hainan")):
    """
    将筛选后的 df 转换为 long 型 dataframe。

    输出格式：
    ID | region | 表型1 | 表型2 | 表型3 | ...

    Population 列会被删除。
    """

    out_list = []

    for region in regions:
        tmp = df.loc[:, region].copy()
        tmp.insert(0, "region", region)
        tmp.insert(0, "ID", df[("meta", "ID")].values)
        out_list.append(tmp)

    return pd.concat(out_list, ignore_index=True)

In [11]:
"""
读入数据
"""
raw_file_path = "/share/home/zhoujb/project/riceDataComb/processData/dataFromScience/phen_from_science.xlsx"
df = load_rice_phenotype_excel(raw_file_path)

In [12]:
"""
获取 Shanghai、Hangzhou、Hainan 三个地域共同存在的数值型表型列。
返回结果仍然保留 MultiIndex 列结构，并保留 meta/Population 和 meta/ID。
"""
df_comm = get_common_numeric_phenotypes(df, regions=("Shanghai", "Hangzhou", "Hainan"))

"""
选择全部品种，或选择某几个亲本对应的所有品种。

populations=None 或 populations="all"：返回全部品种
populations="某个亲本名"：返回该亲本下所有品种
populations=["亲本1", "亲本2"]：返回多个亲本下所有品种
"""

df_sel = select_populations(df_comm, populations=None)

"""
将筛选后的 df 转换为 long 型 dataframe。
输出格式：
ID | region | 表型1 | 表型2 | 表型3 | ...
Population 列会被删除。
"""
df_long = to_region_long(df_sel, regions=("Shanghai", "Hangzhou", "Hainan"))
df_long["ID"] = df_long["ID"].str.replace(r"^MAGIC-", "M-", regex=True)

In [13]:
df_long.to_csv("/share/home/zhoujb/project/riceDataComb/processData/dataFromScience/phen_matrix_1.txt", sep="\t", index=0)

In [14]:
df_long

trait,ID,region,Heading date,Plant height,Culm length,Panicle length,Panicle number,Leaf length,Leaf width,Leaf angle
0,N1-10-10A,Shanghai,103.000000,75.0,57.4,17.6,9.333333,30.1,1.9,13.0
1,N1-10-10B,Shanghai,84.333333,160.9,139.2,21.7,21.333333,37.9,1.9,16.0
2,N1-10-10C,Shanghai,94.666667,197.3,161.5,35.8,10.666667,58.2,2.3,32.0
3,N1-10-10D,Shanghai,99.666667,195.6,165.7,29.9,14.333333,45.5,1.6,27.0
4,N1-10-11A,Shanghai,82.333333,145.5,126.1,19.4,24.000000,31.9,1.4,15.0
...,...,...,...,...,...,...,...,...,...,...
55258,M-9-7E,Hainan,106.000000,174.0,141.8,32.2,21.000000,22.8,1.4,30.0
55259,M-9-8E,Hainan,94.000000,128.5,102.5,26.0,31.000000,39.2,1.7,30.0
55260,M-9-9A,Hainan,96.000000,103.0,82.5,20.5,27.000000,21.9,1.3,30.0
55261,M-9-9B,Hainan,98.000000,138.5,114.5,24.0,19.000000,34.3,1.8,10.0
